# Experiment 4: Numeric Labels + Chat Template Generation

Tests whether `google/gemma-4-E4B-it` needs `tokenizer.apply_chat_template(...)`.

Changes versus Exp1:
- numeric labels
- `max_new_tokens=2`
- greedy decoding
- wraps prompt with chat template when available


In [ ]:
!pip uninstall -y torchaudio torchvision mlx-vlm || true
!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece


In [1]:
MODELS = {
    "gemma_e4b_it": {"model_id": "google/gemma-4-E4B-it"},
    "qwen3_4b_instruct_2507": {"model_id": "Qwen/Qwen3-4B-Instruct-2507"},
    "gemma_e2b_it": {"model_id": "google/gemma-4-E2B-it"},
}

TASKS = ["afqmc", "tnews", "cmnli"]
SPLIT = "validation"
MAX_SAMPLES = 200
DEBUG_N = 3
MAX_NEW_TOKENS = 2


In [2]:
import gc, os, re, time, warnings
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")

def cleanup_memory():
    gc.collect()
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

def clear_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    cleanup_memory()


In [3]:
TASK_SPECS = {
    "afqmc": {
        "label_id_to_name": {"0": "不同", "1": "相同"},
        "label_name_to_id": {"不同": "0", "相同": "1"},
        "choice_text": "0=不同\n1=相同",
        "valid_ids": ["0", "1"],
        "label_id_style": "fixed_manual_numeric",
    },
    "cmnli": {
        "label_id_to_name": {"0": "中立", "1": "蕴含", "2": "矛盾"},
        "label_name_to_id": {"中立": "0", "蕴含": "1", "矛盾": "2"},
        "choice_text": "0=中立\n1=蕴含\n2=矛盾",
        "valid_ids": ["0", "1", "2"],
        "label_id_style": "fixed_manual_numeric",
    },
    "tnews": {
        "label_id_to_name": {
            "0": "故事", "1": "文化", "2": "娱乐", "3": "体育", "4": "财经",
            "5": "房产", "6": "汽车", "7": "教育", "8": "科技", "9": "国际",
            "10": "旅游", "11": "军事", "12": "股票", "13": "农业", "14": "电竞",
        },
        "label_name_to_id": {
            "故事": "0", "文化": "1", "娱乐": "2", "体育": "3", "财经": "4",
            "房产": "5", "汽车": "6", "教育": "7", "科技": "8", "国际": "9",
            "旅游": "10", "军事": "11", "股票": "12", "农业": "13", "电竞": "14",
        },
        "choice_text": (
            "0=故事\n1=文化\n2=娱乐\n3=体育\n4=财经\n5=房产\n6=汽车\n7=教育\n"
            "8=科技\n9=国际\n10=旅游\n11=军事\n12=股票\n13=农业\n14=电竞"
        ),
        "valid_ids": [str(i) for i in range(15)],
        "label_id_style": "fixed_manual_tnews_numeric",
    },
}

def get_task_spec(task):
    return TASK_SPECS[task]

def label_id_to_name(label_id, spec):
    return spec["label_id_to_name"].get(str(label_id), str(label_id))

def pred_id_to_name(pred_id, spec):
    if pred_id == "__invalid__":
        return "__invalid__"
    return spec["label_id_to_name"].get(str(pred_id), "__invalid__")


In [4]:
def build_user_prompt(task, ex, spec):
    if task == "afqmc":
        return f"""只输出一个数字，不要解释。

{spec["choice_text"]}

句子1：{ex["sentence1"]}
句子2：{ex["sentence2"]}

答案："""
    if task == "cmnli":
        return f"""只输出一个数字，不要解释。

{spec["choice_text"]}

前提：{ex["sentence1"]}
假设：{ex["sentence2"]}

答案："""
    if task == "tnews":
        return f"""只输出一个数字，不要解释。

{spec["choice_text"]}

标题：{ex["sentence"]}

答案："""
    raise ValueError(task)

def apply_chat_template_if_available(tokenizer, user_prompt):
    messages = [{"role": "user", "content": user_prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return user_prompt

def normalize_output(text):
    text = str(text).strip().lower()
    for token in ["<bos>", "<eos>", "<pad>", "<start_of_turn>", "<end_of_turn>", "model", "assistant", "user"]:
        text = text.replace(token, " ")
    text = text.replace("答案：", " ").replace("答案:", " ")
    text = text.replace("类别：", " ").replace("类别:", " ")
    text = text.replace("标签：", " ").replace("标签:", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"[。，“”，、；;:：\[\]\(\)（）\"'`]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_pred_id(raw_output, spec):
    text_norm = normalize_output(raw_output)
    if text_norm in spec["valid_ids"]:
        return text_norm
    m = re.search(r"\b\d+\b", text_norm)
    if m:
        candidate = m.group(0)
        if candidate in spec["valid_ids"]:
            return candidate
    for name, label_id in spec["label_name_to_id"].items():
        if name in raw_output:
            return label_id
    return "__invalid__"


In [5]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model

@torch.no_grad()
def generate_answer(tokenizer, model, user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    final_prompt = apply_chat_template_if_available(tokenizer, user_prompt)
    inputs = tokenizer(final_prompt, return_tensors="pt")
    try:
        inputs = inputs.to(model.device)
    except Exception:
        pass
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )
    return decoded.strip(), final_prompt


In [6]:
def evaluate_loaded_model(model_key, model_id, tokenizer, model):
    model_summaries, model_rows = [], []

    for task in TASKS:
        dataset = load_dataset("clue", task, split=SPLIT)
        dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
        spec = get_task_spec(task)

        y_true, y_pred = [], []
        start = time.time()

        for idx, ex in enumerate(tqdm(dataset, desc=f"{model_key}/{task}")):
            user_prompt = build_user_prompt(task, ex, spec)
            raw, final_prompt = generate_answer(tokenizer, model, user_prompt)

            gold_id = str(ex["label"])
            pred_id = extract_pred_id(raw, spec)

            y_true.append(gold_id)
            y_pred.append(pred_id)

            row = {
                "model_key": model_key,
                "model_id": model_id,
                "task": task,
                "split": SPLIT,
                "idx": idx,
                "gold_id": gold_id,
                "gold_name": label_id_to_name(gold_id, spec),
                "raw": repr(raw),
                "raw_normalized": normalize_output(raw),
                "pred_name": pred_id_to_name(pred_id, spec),
                "pred_id": pred_id,
                "used_chat_template": final_prompt != user_prompt,
                "final_prompt_preview": repr(final_prompt[:300]),
            }
            model_rows.append(row)
            if idx < DEBUG_N:
                print(row)

        elapsed = time.time() - start
        invalid_count = sum(p == "__invalid__" for p in y_pred)
        y_pred_for_score = [p if p != "__invalid__" else "-1" for p in y_pred]

        summary = {
            "model_key": model_key,
            "model_id": model_id,
            "task": task,
            "split": SPLIT,
            "samples": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred_for_score),
            "macro_f1": f1_score(y_true, y_pred_for_score, average="macro", zero_division=0),
            "invalid_rate": invalid_count / len(y_pred),
            "invalid_count": invalid_count,
            "seconds": elapsed,
            "samples_per_second": len(y_true) / elapsed if elapsed > 0 else None,
            "label_id_style": spec["label_id_style"],
        }
        print(summary)
        model_summaries.append(summary)

    return model_summaries, model_rows

def evaluate_one_model(model_key, model_cfg):
    model_id = model_cfg["model_id"]
    cleanup_memory()
    print(f"\n=== Loading {model_key}: {model_id} ===")

    tokenizer = None
    model = None
    try:
        tokenizer, model = load_model(model_id)
        summaries, rows = evaluate_loaded_model(model_key, model_id, tokenizer, model)
    finally:
        print(f"Clearing model from memory: {model_key}")
        clear_model(model, tokenizer)

    return summaries, rows


In [7]:
all_summaries, all_rows = [], []

for model_key, model_cfg in MODELS.items():
    summaries, rows = evaluate_one_model(model_key, model_cfg)
    all_summaries.extend(summaries)
    all_rows.extend(rows)

summary_df = pd.DataFrame(all_summaries)
detail_df = pd.DataFrame(all_rows)

summary_df



=== Loading gemma_e4b_it: google/gemma-4-E4B-it ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

gemma_e4b_it/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '不同', 'raw': "'0'", 'raw_normalized': '0', 'pred_name': '不同', 'pred_id': '0', 'used_chat_template': True, 'final_prompt_preview': "'<bos><|turn>user\\n只输出一个数字，不要解释。\\n\\n0=不同\\n1=相同\\n\\n句子1：双十一花呗提额在哪\\n句子2：里可以提花呗额度\\n\\n答案：<turn|>\\n<|turn>model\\n'"}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 1, 'gold_id': '0', 'gold_name': '不同', 'raw': "'1'", 'raw_normalized': '1', 'pred_name': '相同', 'pred_id': '1', 'used_chat_template': True, 'final_prompt_preview': "'<bos><|turn>user\\n只输出一个数字，不要解释。\\n\\n0=不同\\n1=相同\\n\\n句子1：花呗支持高铁票支付吗\\n句子2：为什么友付宝不支持花呗付款\\n\\n答案：<turn|>\\n<|turn>model\\n'"}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 2, 'gold_id': '1', 'gold_name': '相同', 'raw': "'1'", 'raw_normalized': '1', 'pred_n

...

gemma_e2b_it/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'tnews', 'split': 'validation', 'idx': 0, 'gold_id': '2', 'gold_name': '娱乐', 'raw': "'2'", 'raw_normalized': '2', 'pred_name': '娱乐', 'pred_id': '2', 'used_chat_template': True, 'final_prompt_preview': "'<bos><|turn>user\\n只输出一个数字，不要解释。\\n\\n0=故事\\n1=文化\\n2=娱乐\\n3=体育\\n4=财经\\n5=房产\\n6=汽车\\n7=教育\\n8=科技\\n9=国际\\n10=旅游\\n11=军事\\n12=股票\\n13=农业\\n14=电竞\\n\\n标题：江疏影甜甜圈自拍，迷之角度竟这么好看，美吸引一切事物\\n\\n答案：<turn|>\\n<|turn>model\\n'"}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'tnews', 'split': 'validation', 'idx': 1, 'gold_id': '9', 'gold_name': '国际', 'raw': "'11'", 'raw_normalized': '11', 'pred_name': '军事', 'pred_id': '11', 'used_chat_template': True, 'final_prompt_preview': "'<bos><|turn>user\\n只输出一个数字，不要解释。\\n\\n0=故事\\n1=文化\\n2=娱乐\\n3=体育\\n4=财经\\n5=房产\\n6=汽车\\n7=教育\\n8=科技\\n9=国际\\n10=旅游\\n11=军事\\n12=股票\\n13=农业\\n14=电竞\\n\\n标题：以色列大规模空袭开始！伊朗多个军事目标遭遇打击，誓言对等反击\\n\\n答案：<turn|>\\n<|turn>model\\n'"}
{

gemma_e2b_it/cmnli:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '中立', 'raw': "'1'", 'raw_normalized': '1', 'pred_name': '蕴含', 'pred_id': '1', 'used_chat_template': True, 'final_prompt_preview': "'<bos><|turn>user\\n只输出一个数字，不要解释。\\n\\n0=中立\\n1=蕴含\\n2=矛盾\\n\\n前提：新的权利已经足够好了\\n假设：每个人都很喜欢最新的福利\\n\\n答案：<turn|>\\n<|turn>model\\n'"}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 1, 'gold_id': '1', 'gold_name': '蕴含', 'raw': "'2'", 'raw_normalized': '2', 'pred_name': '矛盾', 'pred_id': '2', 'used_chat_template': True, 'final_prompt_preview': "'<bos><|turn>user\\n只输出一个数字，不要解释。\\n\\n0=中立\\n1=蕴含\\n2=矛盾\\n\\n前提：嗯，我不知道，我对他有复杂的感情，嗯，有时候我喜欢他，但同时我也喜欢看到有人打他\\n假设：我在很大程度上喜欢他，但还是喜欢看到有人打他。\\n\\n答案：<turn|>\\n<|turn>model\\n'"}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 2, 'gold_id': '2', 'gold_na

,model_key,model_id,task,split,samples,accuracy,macro_f1,invalid_rate,invalid_count,seconds,samples_per_second,label_id_style
0,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,200,0.515,0.514016,0.000,0,78.524776,2.546967,fixed_manual_numeric
1,gemma_e4b_it,google/gemma-4-E4B-it,tnews,validation,200,0.380,0.340299,0.000,0,107.622495,1.858348,fixed_manual_tnews_numeric
2,gemma_e4b_it,google/gemma-4-E4B-it,cmnli,validation,200,0.545,0.485476,0.000,0,93.804626,2.132091,fixed_manual_numeric
3,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,afqmc,validation,200,0.495,0.482569,0.000,0,61.624415,3.245467,fixed_manual_numeric
4,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,tnews,validation,200,0.270,0.234219,0.000,0,85.203162,2.347331,fixed_manual_tnews_numeric
5,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,cmnli,validation,200,0.665,0.518435,0.000,0,70.147085,2.851152,fixed_manual_numeric
6,gemma_e2b_it,google/gemma-4-E2B-it,afqmc,validation,200,0.670,0.576434,0.000,0,41.681434,4.798299,fixed_manual_numeric
7,gemma_e2b_it,google/gemma-4-E2B-it,tnews,validation,200,0.365,0.301179,0.005,1,60.049298,3.330597,fixed_manual_tnews_numeric
8,gemma_e2b_it,google/gemma-4-E2B-it,cmnli,validation,200,0.480,0.390702,0.000,0,49.601349,4.032148,fixed_manual_numeric


In [ ]:
detail_df.head(20)


In [ ]:
invalid_df = detail_df[detail_df["pred_id"] == "__invalid__"].copy()
print("Invalid count:", len(invalid_df))
invalid_df.head(50)


In [ ]:
detail_df[
    (detail_df["model_key"] == "gemma_e4b_it") &
    (detail_df["task"] == "cmnli")
][["idx", "gold_id", "gold_name", "raw", "raw_normalized", "pred_name", "pred_id", "used_chat_template"]].head(50)


In [ ]:
detail_df.groupby("model_key")["used_chat_template"].mean()


In [ ]:
os.makedirs("results", exist_ok=True)

summary_path = "results/chinese_understanding_summary_v16_exp4_chat_template_generation.csv"
detail_path = "results/chinese_understanding_details_v16_exp4_chat_template_generation.csv"

summary_df.to_csv(summary_path, index=False)
detail_df.to_csv(detail_path, index=False)

print("Saved:")
print(summary_path)
print(detail_path)
